In [1]:
##### Run raster model using country average intensities 

import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import joblib
import rasterio
import gc
import rasterio

In [9]:
##### SET-UP

### Set directories 
# Get the current working directory
cd = Path.cwd().parent.parent 

### Import data 

# Production data (raster)
production_path = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# Capital data (raster)
capital_int_path = f'{cd}/Data/Clean/Predictors/Rasters/country_capital_intensity_tonnes.tif'
capital_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_capital_crosswalk.parquet')

# Labor data (raster)
labor_int_path = f'{cd}/Data/Clean/Predictors/Rasters/country_labor_intensity_tonnes.tif'
labor_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_labor_crosswalk.parquet')

# Geography crosswalk (for aggregating pixels to country)
country_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_country_crosswalk.parquet')

# National data (known totals)
capital_national = pd.read_csv(f'{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv')
labor_national = pd.read_csv(f'{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv')
country_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_country_crosswalk.parquet')

# Set paths 
capital_path = f"{cd}/Results/Raster_model/country_avg_model/capital_USD.tif"
labor_path = f"{cd}/Results/Raster_model/country_avg_model/jobs.tif"

In [10]:
##### CONVERT INTENSITIES TO CAPITAL AND LABOR

def multiply_rasters_and_save(raster1_path, raster2_path, out_path):
    with rasterio.open(raster1_path) as src1, rasterio.open(raster2_path) as src2:
        arr1 = src1.read(1).astype('float32')
        arr2 = src2.read(1).astype('float32')
        profile = src1.profile.copy()

        # handle nodata so it doesn't get multiplied into garbage
        nodata1 = src1.nodata
        nodata2 = src2.nodata
        mask = np.ones(arr1.shape, dtype=bool)
        if nodata1 is not None:
            mask &= (arr1 != nodata1)
        if nodata2 is not None:
            mask &= (arr2 != nodata2)

        out_nodata = -9999.0
        result = np.full(arr1.shape, out_nodata, dtype='float32')
        result[mask] = arr1[mask] * arr2[mask]

        profile.update(dtype='float32', nodata=out_nodata, count=1)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(result, 1)

# Capital
multiply_rasters_and_save(capital_int_path, production_path, capital_path)

# Labor
multiply_rasters_and_save(labor_int_path, production_path, labor_path)


In [14]:
##### Calculate sub-national and national predicted totals 

# Convert raster predictions to df
def raster_to_df(raster_path, value_col="value", drop_nodata=True):
    with rasterio.open(raster_path) as src:
        arr = src.read(1)
        nodata = src.nodata
        transform = src.transform

        nrows, ncols = arr.shape
        rows, cols = np.meshgrid(np.arange(nrows), np.arange(ncols), indexing="ij")

        # Pixel-center coordinates
        xs, ys = rasterio.transform.xy(transform, rows, cols)
        xs = np.array(xs)
        ys = np.array(ys)

    df = pd.DataFrame({
        "x": xs.ravel(),
        "y": ys.ravel(),
        value_col: arr.ravel()
    })

    if drop_nodata and nodata is not None:
        df = df[df[value_col] != nodata].reset_index(drop=True)

    return df

capital_df = raster_to_df(capital_path, value_col="capital_usd")
labor_df = raster_to_df(labor_path, value_col="jobs")

# Merge with GEO_ID
capital_df = capital_df.merge(capital_sub_crosswalk, on=['x', 'y'], how='left')
labor_df = labor_df.merge(labor_sub_crosswalk, on=['x', 'y'], how='left')

# Sum by sub-national region and save
capital_sub_national_predictions = capital_df[['PROJ_ID', 'capital_usd']].groupby('PROJ_ID').sum().reset_index()
labor_sub_national_predictions = labor_df[['PROJ_ID', 'jobs']].groupby('PROJ_ID').sum().reset_index()

capital_sub_national_predictions.to_csv(f"{cd}/Results/Raster_model/country_avg_model/capital_subnational_agg.tif", index=False)
labor_sub_national_predictions.to_csv(f"{cd}/Results/Raster_model/country_avg_model/labor_subnational_agg.tif", index=False)

# Sum by country and save 
capital_df = capital_df.merge(country_crosswalk, on=['x', 'y'], how='left')
labor_df = labor_df.merge(country_crosswalk, on=['x', 'y'], how='left')

capital_national_predictions = capital_df[['GID_0', 'capital_usd']].groupby('GID_0').sum().reset_index()
labor_national_predictions = labor_df[['GID_0', 'jobs']].groupby('GID_0').sum().reset_index()

capital_national_predictions.to_csv(f"{cd}/Results/Raster_model/country_avg_model/capital_national_agg.tif", index=False)
labor_national_predictions.to_csv(f"{cd}/Results/Raster_model/country_avg_model/labor_national_agg.tif", index=False)


In [12]:
#### Rescaling to national totals 

# CONVERT RASTERS TO DF

def raster_to_df(raster_path, value_col):
    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        height, width = arr.shape
        transform = src.transform
        rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
        xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
        xs = np.array(xs)
        ys = np.array(ys)

    return pd.DataFrame({"x": xs, "y": ys, value_col: arr.ravel()})

capital_df = raster_to_df(capital_path, 'capital_USD')
labor_df = raster_to_df(labor_path, 'jobs')

capital_labor_predictions = capital_df.merge(labor_df, on=['x', 'y'], how='outer')

# Re-scale to national

### Step 1: Sum total predicted capital and labor in each country
# Add cross-walk of which country each pixel is in 
country_crosswalk = country_crosswalk.rename(columns={'GID_0': 'ISO3'})
capital_labor_predictions = capital_labor_predictions.merge(country_crosswalk, on=['x', 'y'], how='left')

# Get totals from initial predictions by country
capital_national_predictions = capital_labor_predictions[['ISO3', 'capital_USD']].groupby('ISO3').sum().reset_index()
labor_national_predictions = capital_labor_predictions[['ISO3', 'jobs']].groupby('ISO3').sum().reset_index()


### Step 2: Get re-scaling factor for each country
# Join predicted capital and labor to actual 
capital_national['national_total_capital_USD'] = capital_national['ag_capital_stock_mil_USD_nominal'] * 1e6
capital_national = capital_national.drop(columns=['ag_capital_stock_mil_USD_nominal'])

labor_national['national_total_jobs'] = labor_national['ag_labor_thousands_2020'] * 1e3
labor_national = labor_national.drop(columns=['ag_labor_thousands_2020'])

capital_national_predictions = capital_national_predictions.merge(capital_national[['ISO3', 'national_total_capital_USD']], on='ISO3', how='left')
labor_national_predictions = labor_national_predictions.merge(labor_national[['ISO3', 'national_total_jobs']], on='ISO3', how='left')

# Calculate rescaling factors 
capital_national_predictions['capital_nat_scale'] = capital_national_predictions['national_total_capital_USD'] / capital_national_predictions['capital_USD']
labor_national_predictions['labor_nat_scale'] = labor_national_predictions['national_total_jobs'] / labor_national_predictions['jobs']

# Replace inf/-inf with NaN 
capital_national_predictions['capital_nat_scale'] = capital_national_predictions['capital_nat_scale'].replace([np.inf, -np.inf], np.nan)
labor_national_predictions['labor_nat_scale'] = labor_national_predictions['labor_nat_scale'].replace([np.inf, -np.inf], np.nan)

# Merge back to data 
capital_national_predictions = capital_national_predictions.drop(columns=['capital_USD'])
labor_national_predictions = labor_national_predictions.drop(columns=['jobs'])

capital_labor_predictions = capital_labor_predictions.merge(capital_national_predictions, on='ISO3', how='left')
capital_labor_predictions = capital_labor_predictions.merge(labor_national_predictions, on='ISO3', how='left')


### Step 3: Execute re-scaling
capital_labor_predictions['rescaled_capital_USD'] = capital_labor_predictions['capital_USD'] * capital_labor_predictions['capital_nat_scale']
capital_labor_predictions['rescaled_jobs'] = capital_labor_predictions['jobs'] * capital_labor_predictions['labor_nat_scale']

##### SANITY CHECKS
cap_err_per = (capital_labor_predictions['rescaled_capital_USD'].sum() / capital_labor_predictions['national_total_capital_USD'].sum()) * 100
lab_err_per = (capital_labor_predictions['rescaled_jobs'].sum() / capital_labor_predictions['national_total_jobs'].sum()) * 100

print("Percentage difference between sum of pixels and sum of countries:")
print(f"Capital: {cap_err_per:.2f}")
print(f"Labor: {lab_err_per:.2f}")

##### SAVE FINAL OUTPUTS 
col_to_keep = ['x', 'y', 'rescaled_capital_USD', 'rescaled_jobs']
capital_labor_final = capital_labor_predictions[col_to_keep]

# define function to convert to raster 
def save_column_as_raster(df, value_col, reference_raster_path, output_path):

    # use production raster as reference grid
    with rasterio.open(reference_raster_path) as ref:
        transform = ref.transform
        crs = ref.crs
        height = ref.height
        width = ref.width

    # pivot into 2D grid: rows = y, cols = x
    pivot = df.pivot(index='y', columns='x', values=value_col)

    # match raster row/col order: y descending (north to south), x ascending
    pivot = pivot.sort_index(ascending=False)
    pivot = pivot.reindex(columns=sorted(pivot.columns))

    arr = pivot.values.astype(np.float32)

    if arr.shape != (height, width):
        raise ValueError(
            f"Shape mismatch for {value_col}: got {arr.shape}, expected {(height, width)}"
        )

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(arr, 1)

output_cols = ['rescaled_capital_USD', 'rescaled_jobs']

for col in output_cols:
    out_path = f"{cd}/Results/Raster_model/country_avg_model/{col}.tif"
    save_column_as_raster(capital_labor_final, col, production_path, out_path)
    print(f"Saved {col} -> {out_path}")

Percentage difference between sum of pixels and sum of countries:
Capital: 0.00
Labor: 0.00
Saved rescaled_capital_USD -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/country_avg_model/rescaled_capital_USD.tif
Saved rescaled_jobs -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/country_avg_model/rescaled_jobs.tif
